In [30]:
from langgraph.graph import StateGraph, START, END
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
from typing import TypedDict,Annotated
import os
import operator
from pydantic import BaseModel, Field
from dotenv import load_dotenv
from typing import Literal
from langchain_core.messages import SystemMessage, HumanMessage
from langgraph.checkpoint.memory import MemorySaver

load_dotenv()

True

In [31]:
endpoint=HuggingFaceEndpoint(
    repo_id="meta-llama/Llama-3.1-8B-Instruct",
    huggingfacehub_api_token=os.getenv("HUGGINGFACEHUB_API_TOKEN"),
    task="text-generation"
    )
model=ChatHuggingFace(llm=endpoint)

In [32]:
class JokeState(TypedDict):
    topic:str
    joke:str
    explaination:str

In [33]:
def generate_joke(state:JokeState):
    prompt=f"""generate a joke on the topic {state['topic']}"""
    response=model.invoke(prompt).content
    return {"joke":response}

In [34]:
def generate_explaination(state:JokeState):
    prompy=f"""explain the joke {state['joke']}"""
    response=model.invoke(prompy).content
    return {"explaination":response}

In [35]:
graph=StateGraph(JokeState)
graph.add_node("generate_joke",generate_joke)
graph.add_node("generate_explaination",generate_explaination)

graph.add_edge(START,"generate_joke")
graph.add_edge("generate_joke","generate_explaination")
graph.add_edge("generate_explaination",END)

checkpointer=MemorySaver()

workflow=graph.compile(checkpointer=checkpointer)

In [36]:
config1 = {'configurable': {'thread_id': '1'}}
initial_state={
    "topic":"Money"
}

workflow.invoke(initial_state,config=config1)


{'topic': 'Money',
 'joke': "Here's one:\n\nWhy did the dollar bill go to the doctor?\n\nBecause it was feeling a little flat! (get it?)",
 'explaination': 'The joke is a play on words. "Flat" has a double meaning here:\n\n1. A dollar bill is a flat piece of paper, so it\'s literally feeling flat because it\'s a 2D object.\n2. When someone is "feeling flat," it\'s an idiomatic expression meaning they\'re feeling unwell or depressed.\n\nThe joke is funny because it\'s a clever and unexpected pun, combining the literal meaning of "flat" (the bill is a flat piece of paper) with the idiomatic expression "feeling flat" (meaning feeling unwell). The punchline relies on a quick mental switch between the two meanings, creating a sense of surprise and delight.'}

In [37]:
workflow.get_state(config1)

StateSnapshot(values={'topic': 'Money', 'joke': "Here's one:\n\nWhy did the dollar bill go to the doctor?\n\nBecause it was feeling a little flat! (get it?)", 'explaination': 'The joke is a play on words. "Flat" has a double meaning here:\n\n1. A dollar bill is a flat piece of paper, so it\'s literally feeling flat because it\'s a 2D object.\n2. When someone is "feeling flat," it\'s an idiomatic expression meaning they\'re feeling unwell or depressed.\n\nThe joke is funny because it\'s a clever and unexpected pun, combining the literal meaning of "flat" (the bill is a flat piece of paper) with the idiomatic expression "feeling flat" (meaning feeling unwell). The punchline relies on a quick mental switch between the two meanings, creating a sense of surprise and delight.'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1b1b61-f215-6b5b-8002-036d679031e8'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-09-16T10:05:

In [38]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'Money', 'joke': "Here's one:\n\nWhy did the dollar bill go to the doctor?\n\nBecause it was feeling a little flat! (get it?)", 'explaination': 'The joke is a play on words. "Flat" has a double meaning here:\n\n1. A dollar bill is a flat piece of paper, so it\'s literally feeling flat because it\'s a 2D object.\n2. When someone is "feeling flat," it\'s an idiomatic expression meaning they\'re feeling unwell or depressed.\n\nThe joke is funny because it\'s a clever and unexpected pun, combining the literal meaning of "flat" (the bill is a flat piece of paper) with the idiomatic expression "feeling flat" (meaning feeling unwell). The punchline relies on a quick mental switch between the two meanings, creating a sense of surprise and delight.'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1b1b61-f215-6b5b-8002-036d679031e8'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-09-16T10:05

Time Travel


In [39]:
workflow.get_state({"configurable":{"thread_id":"1","checkpoint_id":"1f1b1b29-30be-6195-8000-bda4ad152e44"}})

StateSnapshot(values={}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_id': '1f1b1b29-30be-6195-8000-bda4ad152e44'}}, metadata=None, created_at=None, parent_config=None, tasks=(), interrupts=())

In [40]:
# workflow.invoke(None,{"configurable":{"thread_id":"1","checkpoint_id":"1f1b1b29-30be-6195-8000-bda4ad152e44"}})

In [41]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'Money', 'joke': "Here's one:\n\nWhy did the dollar bill go to the doctor?\n\nBecause it was feeling a little flat! (get it?)", 'explaination': 'The joke is a play on words. "Flat" has a double meaning here:\n\n1. A dollar bill is a flat piece of paper, so it\'s literally feeling flat because it\'s a 2D object.\n2. When someone is "feeling flat," it\'s an idiomatic expression meaning they\'re feeling unwell or depressed.\n\nThe joke is funny because it\'s a clever and unexpected pun, combining the literal meaning of "flat" (the bill is a flat piece of paper) with the idiomatic expression "feeling flat" (meaning feeling unwell). The punchline relies on a quick mental switch between the two meanings, creating a sense of surprise and delight.'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1b1b61-f215-6b5b-8002-036d679031e8'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-09-16T10:05

Updating State

In [42]:
workflow.update_state({"configurable":{"thread_id":"1","checkpoint_id":"1f1b1b4e-5a53-6a0a-8001-78ce418c6546","checkpoint_ns":""}},{"topic":"Samosa"})

{'configurable': {'thread_id': '1',
  'checkpoint_ns': '',
  'checkpoint_id': '1f1b1b61-f361-614f-8000-f55afd6ead54'}}

In [43]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'Samosa'}, next=('generate_joke',), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1b1b61-f361-614f-8000-f55afd6ead54'}}, metadata={'source': 'update', 'step': 0, 'parents': {}}, created_at='2026-09-16T10:05:19.099732+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1b1b4e-5a53-6a0a-8001-78ce418c6546'}}, tasks=(PregelTask(id='04be96d4-8a32-9611-adaa-d747f2604527', name='generate_joke', path=('__pregel_pull', 'generate_joke'), error=None, interrupts=(), state=None, result=None),), interrupts=()),
 StateSnapshot(values={'topic': 'Money', 'joke': "Here's one:\n\nWhy did the dollar bill go to the doctor?\n\nBecause it was feeling a little flat! (get it?)", 'explaination': 'The joke is a play on words. "Flat" has a double meaning here:\n\n1. A dollar bill is a flat piece of paper, so it\'s literally feeling flat because it\'s a 2D object.\n2. When someone is "feeling flat,"